In [49]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score , root_mean_squared_error
import joblib
import pickle

In [26]:
df["Date"] = pd.to_datetime(df["Date"], format="mixed")

df = df.dropna(subset=["year","month","day"])
# labeliser les données
df["Service"] = df["Service"].astype("category").cat.codes
df["Gare de départ"] = df["Gare de départ"].astype("category").cat.codes
df["Gare d'arrivée"] = df["Gare d'arrivée"].astype("category").cat.codes

# Baseline (reference minimal)
baseline_pred = np.full(len(y_test), y_train.mean())
baseline_RMSE = root_mean_squared_error(y_test, baseline_pred)
baseline_MAE = mean_absolute_error(y_test, baseline_pred)

X = df[["Gare de départ","Gare d'arrivée","Durée moyenne du trajet","month","year"]]
y = df["Retard moyen de tous les trains à l'arrivée"]

# Diviser les X (les features ) et y (valeurs que le modele predit)
X_train, X_test, y_train, y_test = train_test_split(X,y , random_state=104,test_size=0.15, shuffle=True)

#Modele lineaire
reg = LinearRegression()
reg.fit(X_train,y_train)
y_pred = reg.predict(X_test)

# evaluation du modele lineaires
R2 = r2_score(y_test,y_pred)
RMSE = root_mean_squared_error(y_test,y_pred)
MAE = mean_absolute_error(y_test,y_pred)

# print('X_train : ')
# print(X_train.head())
# print('')
# print('X_test : ')
# print(X_test.head())
# print('')
# print('y_train : ')
# print(y_train.head())
# print('')
# print('y_test : ')
# print(y_test.head())


### Model Interpretation
I chose the Linear Regression model as a baseline because it is simple, interpretable, and allows us to understand the direct relationship between journey characteristics and delay duration.
### Features used:
Departure station, arrival station: the route directly influences delay likelihood
Average journey duration: longer journeys may accumulate more delays
Month and year: seasonal and temporal patterns affect punctuality

### Target: 
Average delay of all trains upon arrival was chosen because it best represents the passenger experience — what matters is arriving on time, not departing on time.
    
### Metrics results:
R²: the model explains X% of the variance in delays. A value close to 1 means the model predicts well, close to 0 means it struggles
RMSE: on average the model is off by X minutes, penalizing large errors more
MAE: on average the absolute error is X minutes, easier to interpret in real terms

### Limitations: 
Linear regression assumes a straight-line relationship between inputs and output. In reality, delays are influenced by complex non-linear factors such as strikes, weather, or infrastructure failures that a linear model cannot capture well. A tree-based model like Random Forest would likely perform better.


# Load the dataset into the variable 'df'

In [27]:
df = pd.read_csv('cleaned_dataset.csv')

# Display the first few rows to make sure it loaded correctly

In [28]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11896 entries, 0 to 11895
Data columns (total 27 columns):
 #   Column                                                                                       Non-Null Count  Dtype  
---  ------                                                                                       --------------  -----  
 0   Unnamed: 0                                                                                   11896 non-null  int64  
 1   Date                                                                                         11896 non-null  object 
 2   Service                                                                                      11896 non-null  object 
 3   Gare de départ                                                                               11837 non-null  object 
 4   Gare d'arrivée                                                                               11837 non-null  object 
 5   Durée moyenne du trajet         

# 1. Encode the text columns

In [29]:
le = LabelEncoder()
df['Service'] = le.fit_transform(df['Service'])
df['Gare de départ'] = le.fit_transform(df['Gare de départ'])
df["Gare d'arrivée"] = le.fit_transform(df["Gare d'arrivée"])

# 2. Handle missing values
# Dropping rows where the date features are missing to avoid scikit-learn errors

In [30]:
df = df.dropna(subset=['year', 'month', 'day'])

# Create Day of the Week

In [31]:
df['full_date'] = pd.to_datetime(df[['year', 'month', 'day']])
df['day_of_week'] = df['full_date'].dt.dayofweek
df = df.drop(columns=['full_date'])

# 3. Define Features (X) and Target (y)
# Using the encoded categorical variables and date features as inputs

In [32]:
features = ['Service', 'Gare de départ', "Gare d'arrivée", 'year', 'month']
X = df[features]

# Predicting the continuous delay duration in minutes

In [33]:
y = df["Retard moyen de tous les trains à l'arrivée"]

# 4. Train/Test Split
# Creating training data and test data as outlined in your project flowchart

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train the Model
# max_depth limits how deep the tree grows to help prevent overfitting

In [35]:
model = DecisionTreeRegressor(max_depth=10, random_state=42)
model.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_lea

# 6. Evaluate the Model

In [22]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Decision Tree Model Evaluation ---")
print(f"RMSE: {rmse:.2f} minutes")
print(f"MAE:  {mae:.2f} minutes")
print(f"R²:   {r2:.2f}")

--- Decision Tree Model Evaluation ---
RMSE: 10.68 minutes
MAE:  2.63 minutes
R²:   0.02


# 7. Export the Model
# Saving the trained model file for dashboard integration

In [38]:
joblib.dump(model, 'model.joblib')
# print("\nModel successfully saved as 'model.joblib'.")
with open("data.pckl","wb") as p:
    pickle.dump(reg,p)
    